#### Quiz 1
##### NLP, Autumn 2026

This is an *open-book* test. You may consult class notes, your own notes or lab notebooks only.    
**Time limt: 90+10 minutes**    
**Maximum marks = 50**

#### Q1. (Regular expression)

1.1 Write a **regular expression** for the following date formats only.
1. xx-xx-xxxx (03-11-2020)
2. xx.xx.xxxx (06.13.2020)
3. xx/xx/xxxx (23/10/2020)    

The first two digits could represnt day or month. *mm.dd.yyyy* format is followed in European countries.
This will allow dates like **51-22-2012**. Complete the following function to check thet the dates are all in the correct range: the month should be a number betwen 1 and 12 (both included) and the day should be in the appropriate range, depending on the month. For example, if the month is June (06), the day should be in the range 1-30. Note, February is special. Your function must use the same **re** that you created in the first part. The input to the function is a *string*.

**13 marks (4+9)**

In [ ]:
def check_leap_year(year):
    """Method to check if a year is a leap year
       Returns:
        1 if leap year
        0(otherwise) if not leap year
    """
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

def days_in_month(month, year):
    """Method that returns the number of days in a given month for a specific year.
    """
    if month == 2:
        return 29 if check_leap_year(year) else 28
    elif month in [4, 6, 9, 11]:
        return 30
    else:
        return 31

def check_date(date_str):
    import re
    pattern = r"(\d{2})([./-])(\d{2})\2(\d{4})" # Use the re for date pattern.

    # Check if the input string is in one of the allowed date formats
    if not re.fullmatch(pattern, date_str):
        return ("Incorrect date format")
    print(date_str)

    part1 = int(re.fullmatch(pattern, date_str).group(1))
    part2 = int(re.fullmatch(pattern, date_str).group(3))
    year = int(re.fullmatch(pattern, date_str).group(4))

    is_valid_date = False

    # Try to validate assuming part1 is month, part2 is day i.e (MM.DD.YYYY)
    if 1 <= part1 <= 12: # Checks if part1 is a valid month
        month = part1
        day = part2
        if 1 <= day <= days_in_month(month, year):
            is_valid_date = True

    # Try to validate with reverse syntax assuming part1 is day, part2 is month i.e (DD.MM.YYYY)
    if not is_valid_date and 1 <= part2 <= 12: # Checks if part2 is a valid month
        month = part2
        day = part1
        if 1 <= day <= days_in_month(month, year):
            is_valid_date = True

    if is_valid_date:
        return "Okay"
    else:
        return "Not okay"

In [ ]:
# Run test
print(check_date("03-11-2020"))
print(check_date("06.13.2020"))
print(check_date("23/10/2020"))

03-11-2020
Okay
06.13.2020
Okay
23/10/2020
Okay


#### Question 2  
Given the following short movie reviews, each labelled with a genre, either
**comedy** or **action**:
1. fun, couple, love, love : comedy
2. furious, shoot, shoot, fun :action
3. couple, fly, fast, fun, fun :comedy
4. fast, kids, fight, laugh :comedy
5. fly, fast, shoot, love, run :action
6. family, laugh, run, love  :comedy
7. fast, run, shoot, love, fast : action
8. run, furious, fast, fun, kids: action

Consider two new documents D1 and D2:
1. D1: family, fun, love, run
2. D2: fast, kids, shoot, fly

Compute the most likely class for D1 and D2. Assume a naive Bayes classifier and use
add-1 smoothing for the likelihoods.

1. You should first build the *vocabulary* V consisting of the **set** of words. Recall that a set has no duplications.
2. For each word *w* in the V find the probability *P*(*w* | comedy), the probability that the word *w* occurs in a comedy. You should use add-1 smoothing.
3. Now use Naive Bayes classifier to decide what classes we put D1 and D2 into.

**25 marks**  

**Answer Q2**  



In [ ]:
# First step we define the dataset
reviews = [
    ("fun couple love love".split(), "comedy"),
    ("furious shoot shoot fun".split(), "action"),
    ("couple fly fast fun fun".split(), "comedy"),
    ("fast kids fight laugh".split(), "comedy"),
    ("fly fast shoot love run".split(), "action"),
    ("family laugh run love".split(), "comedy"),
    ("fast run shoot love fast".split(), "action"),
    ("run furious fast fun kids".split(), "action")
]

document_d1 = "family fun love run".split()
document_d2 = "fast kids shoot fly".split()

# Second step we build the Vocabulary (V)
vocabulary = set()
for words, _ in reviews:
    for word in words:
        vocabulary.add(word)

v = list(vocabulary)
v_size = len(v)

# Third step we calculate prior probabilities P(class)
total_documents = len(reviews)
comedy_docs = sum(1 for _, label in reviews if label == "comedy")
action_docs = sum(1 for _, label in reviews if label == "action")

p_comedy = comedy_docs / total_documents
p_action = action_docs / total_documents

# Fourth step we calculate the conditional probabilities P(word|class) with add-1 smoothing
from collections import defaultdict

# Count word occurrences in each class
word_counts_comedy = defaultdict(int)
word_counts_action = defaultdict(int)
total_words_comedy = 0
total_words_action = 0

for words, label in reviews:
    for word in words:
        if label == "comedy":
            word_counts_comedy[word] += 1
            total_words_comedy += 1
        else:
            word_counts_action[word] += 1
            total_words_action += 1

# Calculate conditional probabilities with add-1 smoothing
prob_word_given_comedy = {}
prob_word_given_action = {}

for word in v:
    prob_word_given_comedy[word] = (word_counts_comedy[word] + 1) / (total_words_comedy + v_size)

for word in v:
    prob_word_given_action[word] = (word_counts_action[word] + 1) / (total_words_action + v_size)

# Last step do classification for documents D1 and D2 using Naive Bayes
import math

def classify_document(document, p_comedy, p_action, prob_word_given_comedy, prob_word_given_action, v):
    log_prob_comedy = math.log(p_comedy)
    log_prob_action = math.log(p_action)

    for word in document:
        if word in v:
            log_prob_comedy += math.log(prob_word_given_comedy.get(word, 1 / (total_words_comedy + v_size)))
            log_prob_action += math.log(prob_word_given_action.get(word, 1 / (total_words_action + v_size)))
        else:
            # Handle words not in vocabulary with add-1 smoothing and full vocabulary, though less common occurrence.
            log_prob_comedy += math.log(1 / (total_words_comedy + v_size))
            log_prob_action += math.log(1 / (total_words_action + v_size))

    if log_prob_comedy > log_prob_action:
        return "comedy"
    else:
        return "action"

# Run classification check for D1
predicted_class_d1 = classify_document(document_d1, p_comedy, p_action, prob_word_given_comedy, prob_word_given_action, v)
print(f"D1: {document_d1} -> Predicted Class: {predicted_class_d1}")

# Run classification check for D2
predicted_class_d2 = classify_document(document_d2, p_comedy, p_action, prob_word_given_comedy, prob_word_given_action, v)
print(f"D2: {document_d2} -> Predicted Class: {predicted_class_d2}")

D1: ['family', 'fun', 'love', 'run'] -> Predicted Class: comedy
D2: ['fast', 'kids', 'shoot', 'fly'] -> Predicted Class: action


#### Question 3

The name-gender classification was done using Naive Bayes. Suppose we want to use **logistic regression (LOGIT)** for the same problem. Again we will take *first* and the *last* letters in the names as features. In order to apply LOGIT we have to convert the features to appropriate vectors. Use *one-hot-encoding* for each feature. Complete the following function. It takes a string of uppercase letters (*name*) as input and outputs 0-1 array of appropritae dimension. Complete the following function.

**12 marks**  

In [ ]:
def onehot_name_encoding(name):
    name = name.lower()
    onehot_vec = [0] * 52 # 26 for first letter (a-z), 26 for last letter (a-z)

    if len(name) > 0:
        # We do first letter encoding i.e (index 0-25 for 'a' to 'z')
        first_char = name[0]
        if 'a' <= first_char <= 'z':
            onehot_vec[ord(first_char) - ord('a')] = 1

        # We do encoding of last letter (index 26-51 for 'a' to 'z')
        last_char = name[-1]
        if 'a' <= last_char <= 'z':
            onehot_vec[26 + (ord(last_char) - ord('a'))] = 1

    return onehot_vec

In [ ]:
onehot_name_encoding('Christian')

[0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]